## Imports and Configuration

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, inspect
from snowflake.sqlalchemy import URL
from urllib.parse import quote_plus

# --- POSTGRES CONFIG ---
PG_DB = {
    "user": "<user_name>",
    "pass": "<password>",
    "host": "<host_name>",
    "port": "5432",
    "db": "<database_name>"
}

# --- SNOWFLAKE CONFIG ---
SNOW_CONF = {
    "account": "<snowflake_account_identifier>", 
    "user": "<user_name>",
    "password": "<password>",
    "database": "<database_name>",
    "schema": "<schema_name>",
    "warehouse": "<warehouse_name>",
    "role": "<role_name>"
}

print("Libraries imported and config set")

## Create Engines and Test Connections

In [ ]:

# Postgres Engine
pg_pass = quote_plus(PG_DB['pass'])
pg_url = f"postgresql://{PG_DB['user']}:{pg_pass}@{PG_DB['host']}:{PG_DB['port']}/{PG_DB['db']}"
pg_engine = create_engine(pg_url)

# Snowflake Engine
snow_url = URL(
    account=SNOW_CONF['account'],
    user=SNOW_CONF['user'],
    password=SNOW_CONF['password'],
    database=SNOW_CONF['database'],
    schema=SNOW_CONF['schema'],
    warehouse=SNOW_CONF['warehouse'],
    role=SNOW_CONF['role']
)
snow_engine = create_engine(snow_url)

# Test
try:
    with pg_engine.connect() as conn:
        print("Postgres Connection Successful")
    with snow_engine.connect() as conn:
        print("Snowflake Connection Successful")
except Exception as e:
    print(f"Connection Error: {e}")

## Inspect Tables

In [ ]:
inspector = inspect(pg_engine)
pg_tables = inspector.get_table_names(schema='public') # Pulling from dvdrental public schema in postgresql

print(f"Found {len(pg_tables)} tables in SQL Server: {pg_tables}")

## The Migration Loop

In [ ]:
for table in pg_tables:
    try:
        if table == 'film':
            print(f"Skipping {table} (Excluded intentionally)")
            continue

        print(f"--- Migrating: {table} to Snowflake ---")
        
        # EXTRACT
        df = pd.read_sql_table(table, pg_engine, schema='public')

        if df.empty:
            print(f"Table {table} is empty. Skipping...")
            continue
        
        # TRANSFORM: Prepend prefix (Snowflake will uppercase this)
        destination_name = f"stg_{table}"
        
        # LOAD
        # Snowflake to_sql is slightly slower than local DBs because it's cloud-based.
        # For huge datasets, Snowflake prefers "Stage & Copy" via S3, but for 
        # small/medium tables, to_sql works fine.
        print(f"Writing {table} to Snowflake ({len(df)} rows)...")
        df.to_sql(
            destination_name, 
            snow_engine, 
            if_exists='replace', 
            index=False,
            chunksize=5000 # Snowflake handles large chunks well
        )
        print(f"Successfully moved {table} -> {destination_name.upper()}")
        
    except Exception as e:
        print(f"Error with {table}: {e}")